# Phase 2 — Data analysis and visual diagnostics

This notebook audits the tomato and carrot price exports, creates leakage-safe *as-of* forecasting features, makes a chronological train/validation/test split, and exports the eight required 300-DPI figures.

**Leakage rule:** only deterministic historical features use observations available on or before an origin date. Learned transformations, PCA, and model interpretation are fitted using the training partition only.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', context='notebook')

# Supports running from the repository root or this notebook's directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

def save_figure(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved: {path.relative_to(PROJECT_ROOT)}')

def add_title(fig, title):
    fig.suptitle(title, y=1.02, fontsize=14, fontweight='bold')


## 1. Load and clean the raw exports

Equal-price duplicates are collapsed. Rows with the same record identity but different prices are excluded from modelling and retained separately for review.

In [ ]:
FILES = ['tangabid_market_prices.csv', 'tangabid_market_prices_carrots.csv']
raw = pd.concat([pd.read_csv(RAW_DIR / name) for name in FILES], ignore_index=True)
raw['observed_date'] = pd.to_datetime(raw['observed_date'], errors='coerce')
raw['price'] = pd.to_numeric(raw['price'], errors='coerce')
raw['product'] = raw['product'].replace({'Tomatos': 'Tomato'}).str.strip()
for column in ['unit', 'channel', 'location', 'region', 'currency', 'source']:
    raw[column] = raw[column].astype('string').str.strip()

required = ['observed_date', 'product', 'unit', 'channel', 'location', 'region', 'price', 'currency']
invalid = raw[raw[required].isna().any(axis=1) | raw['price'].le(0)].copy()
valid = raw.drop(index=invalid.index).copy()

IDENTITY = ['product', 'region', 'location', 'unit', 'currency', 'channel',
            'observed_date', 'variety', 'grade']
n_prices = valid.groupby(IDENTITY, dropna=False)['price'].nunique()
conflict_index = n_prices[n_prices.gt(1)].index
row_index = pd.MultiIndex.from_frame(valid[IDENTITY])
conflicts = valid.loc[row_index.isin(conflict_index)].copy()
clean = valid.loc[~row_index.isin(conflict_index)].drop_duplicates(IDENTITY).copy()
clean['market_id'] = clean['region'] + ' | ' + clean['location']
clean = clean.sort_values(['product', 'market_id', 'observed_date']).reset_index(drop=True)

print(f'Raw rows: {len(raw):,}')
print(f'Invalid rows removed: {len(invalid):,}')
print(f'Conflicting rows quarantined: {len(conflicts):,}')
print(f'Clean observations: {len(clean):,}')
display(clean.groupby('product').agg(rows=('price', 'size'), markets=('market_id', 'nunique'),
                                     first_date=('observed_date', 'min'), last_date=('observed_date', 'max')))

## 2. Figure 1 — Missingness heatmap

A fixed random sample keeps the row-level heatmap readable while preserving the column-level missingness pattern.

In [ ]:
heatmap_sample = raw.sample(n=min(250, len(raw)), random_state=RANDOM_STATE).sort_values('observed_date')
fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(heatmap_sample.isna(), cbar=False, yticklabels=False, cmap=['#2a9d8f', '#e76f51'], ax=ax)
ax.set_xlabel('Raw column')
ax.set_ylabel('Sampled raw records')
add_title(fig, 'Figure 1. Missingness heatmap (250 sampled raw records)')
save_figure(fig, '01_missingness_heatmap.png')

display((raw.isna().mean().mul(100).rename('missing_pct').to_frame().round(2)))

## 3. Exact-date forecasting labels and as-of features

Targets exist only when the same crop-market series has an observed price exactly 1, 2, or 3 days after the forecast origin. Previous observations are not assumed to be previous calendar days.

In [ ]:
SERIES = ['product', 'market_id']
history = clean.copy()
for lag in range(1, 4):
    history[f'lag_{lag}_price'] = history.groupby(SERIES)['price'].shift(lag)
    previous_date = history.groupby(SERIES)['observed_date'].shift(lag)
    history[f'lag_{lag}_age_days'] = (history['observed_date'] - previous_date).dt.days

def add_rolling_features(group):
    group = group.sort_values('observed_date').copy()
    rolling = group.rolling('7D', on='observed_date', closed='both')['price']
    group['trailing_7d_mean'] = rolling.mean().to_numpy()
    group['trailing_7d_std'] = rolling.std().to_numpy()
    group['trailing_7d_count'] = rolling.count().to_numpy()
    return group

history = history.groupby(SERIES, group_keys=False).apply(add_rolling_features)
history['previous_relative_change'] = (history['price'] / history['lag_1_price']) - 1
history['month_sin'] = np.sin(2 * np.pi * history['observed_date'].dt.month / 12)
history['month_cos'] = np.cos(2 * np.pi * history['observed_date'].dt.month / 12)
history['day_of_week'] = history['observed_date'].dt.dayofweek

target_lookup = clean[SERIES + ['observed_date', 'price']].rename(
    columns={'observed_date': 'target_date', 'price': 'target_price'}
)
examples = []
for horizon in (1, 2, 3):
    part = history.copy()
    part['horizon'] = horizon
    part['target_date'] = part['observed_date'] + pd.Timedelta(days=horizon)
    part = part.merge(target_lookup, on=SERIES + ['target_date'], how='left')
    examples.append(part.dropna(subset=['target_price']))
dataset = pd.concat(examples, ignore_index=True).sort_values('observed_date').reset_index(drop=True)
print(f'Observed origin-target pairs: {len(dataset):,}')
display(dataset.groupby(['product', 'horizon']).size().rename('pairs').unstack(fill_value=0))

## 4. Chronological split, target-date purge, and training eligibility

All markets on an origin date belong to the same partition. Training rows whose labels reach the validation period and validation rows whose labels reach the test period are removed.

In [ ]:
origin_dates = np.array(sorted(dataset['observed_date'].unique()))
train_end = pd.Timestamp(origin_dates[int(np.floor(len(origin_dates) * 0.70)) - 1])
validation_end = pd.Timestamp(origin_dates[int(np.floor(len(origin_dates) * 0.85)) - 1])

train = dataset[(dataset['observed_date'] <= train_end) & (dataset['target_date'] <= train_end)].copy()
validation = dataset[(dataset['observed_date'] > train_end) & (dataset['observed_date'] <= validation_end) &
                     (dataset['target_date'] <= validation_end)].copy()
test = dataset[dataset['observed_date'] > validation_end].copy()

eligibility = train.groupby(SERIES + ['horizon']).size().rename('training_pairs')
eligible_index = eligibility[eligibility.ge(20)].index
def keep_eligible(frame):
    return frame.loc[pd.MultiIndex.from_frame(frame[SERIES + ['horizon']]).isin(eligible_index)].copy()

train, validation, test = map(keep_eligible, (train, validation, test))
print(f'Train ends: {train_end.date()} | Validation ends: {validation_end.date()}')
display(pd.DataFrame({'rows': [len(train), len(validation), len(test)]}, index=['train', 'validation', 'test']))
assert train['target_date'].le(train_end).all()
assert validation['target_date'].le(validation_end).all()
assert train['observed_date'].lt(train['target_date']).all()


## 5. Figures 2–3 — Training-target distributions and price outliers

These diagnostics use the training partition only because they may influence modelling choices.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)
for axis, horizon in zip(axes, (1, 2, 3)):
    subset = train[train['horizon'].eq(horizon)]
    sns.histplot(data=subset, x='target_price', hue='product', bins=30, stat='density',
                 common_norm=False, element='step', fill=True, alpha=.25, ax=axis)
    axis.set_title(f'{horizon}-day target')
    axis.set_xlabel('Future observed price (RWF/kg)')
add_title(fig, 'Figure 2. Training target-price distributions by forecast horizon')
fig.tight_layout()
save_figure(fig, '02_target_distribution.png')

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=train.drop_duplicates(SERIES + ['observed_date']), x='product', y='price',
            hue='product', legend=False, showfliers=True, ax=ax)
ax.set(xlabel='Crop', ylabel='Current observed price (RWF/kg)')
add_title(fig, 'Figure 3. Training-price outlier boxplots by crop')
save_figure(fig, '03_outlier_boxplots.png')

## 6. Figures 4–5 — Correlation and collinearity diagnostics

Spearman correlation is used because the price variables are skewed. Only training data are used.

In [ ]:
NUMERIC_FEATURES = ['price', 'lag_1_price', 'lag_2_price', 'lag_3_price',
                    'lag_1_age_days', 'lag_2_age_days', 'lag_3_age_days',
                    'trailing_7d_mean', 'trailing_7d_std', 'trailing_7d_count',
                    'previous_relative_change', 'month_sin', 'month_cos',
                    'day_of_week', 'horizon']
corr = train[NUMERIC_FEATURES].corr(method='spearman')
mask = np.tril(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, cmap='vlag', center=0, vmin=-1, vmax=1, square=True,
            linewidths=.4, cbar_kws={'label': 'Spearman correlation'}, ax=ax)
add_title(fig, 'Figure 4. Upper-triangle correlation heatmap of training features')
save_figure(fig, '04_correlation_heatmap.png')

pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
         .rename('correlation').reset_index().rename(columns={'level_0': 'feature_x', 'level_1': 'feature_y'}))
pairs = pairs[pairs['correlation'].abs().ge(.85)].sort_values('correlation', key=lambda s: s.abs(), ascending=False)
top_pairs = pairs.head(6).reset_index(drop=True)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (_, pair) in zip(axes.flat, top_pairs.iterrows()):
    sns.scatterplot(data=train, x=pair['feature_x'], y=pair['feature_y'], hue='product',
                    alpha=.55, s=22, linewidth=0, ax=ax)
    ax.set_title(f"ρ = {pair['correlation']:.2f}")
for ax in axes.flat[len(top_pairs):]:
    ax.axis('off')
if top_pairs.empty:
    axes.flat[0].text(.5, .5, 'No feature pairs reached |ρ| ≥ 0.85.', ha='center', va='center')
    axes.flat[0].axis('off')
add_title(fig, 'Figure 5. Most collinear training-feature pairs (|Spearman ρ| ≥ 0.85)')
fig.tight_layout()
save_figure(fig, '05_collinear_scatterplots.png')

## 7. Figures 7–8 — Transformation and PCA diagnostics

Log views are diagnostic only; the main tree models retain unscaled values. PCA uses median imputation and standardization fitted on training features only.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_variables = [('price', 'Current price'), ('target_price', 'Future target price')]
for row, (column, label) in enumerate(plot_variables):
    sns.histplot(train[column], bins=35, kde=True, ax=axes[row, 0], color='#457b9d')
    axes[row, 0].set_title(f'Raw {label}')
    sns.histplot(np.log1p(train[column]), bins=35, kde=True, ax=axes[row, 1], color='#e76f51')
    axes[row, 1].set_title(f'log1p({label})')
add_title(fig, 'Figure 7. Transformation diagnostic grid (training data)')
fig.tight_layout()
save_figure(fig, '07_transformation_grid.png')

pca_input = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
X_numeric_train = pca_input.fit_transform(train[NUMERIC_FEATURES])
pca = PCA(random_state=RANDOM_STATE).fit(X_numeric_train)
explained = pca.explained_variance_ratio_
cumulative = explained.cumsum()
components = np.arange(1, len(explained) + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(components, explained, alpha=.65, label='Individual explained variance')
ax.plot(components, cumulative, marker='o', color='#e76f51', label='Cumulative variance')
ax.axhline(.90, color='gray', linestyle='--', linewidth=1, label='90% / 95% thresholds')
ax.axhline(.95, color='gray', linestyle='--', linewidth=1)
ax.set(xlabel='Principal component', ylabel='Explained variance ratio', ylim=(0, 1.05))
ax.legend(loc='center right')
add_title(fig, 'Figure 8. PCA scree plot from standardized training features')
save_figure(fig, '08_pca_scree_plot.png')
print(f"Components needed for 90% variance: {np.searchsorted(cumulative, .90) + 1}")
print(f"Components needed for 95% variance: {np.searchsorted(cumulative, .95) + 1}")

## 8. Figure 6 — Validation-set permutation importance

The preprocessor and XGBoost candidate are fitted on `X_train` only. Importance is then measured by how much validation MAE worsens after permuting one original input column at a time. The test partition is not used.

In [ ]:
CATEGORICAL_FEATURES = ['product', 'market_id']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X_train, y_train = train[FEATURES], train['target_price']
X_validation, y_validation = validation[FEATURES], validation['target_price']

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True))]), NUMERIC_FEATURES),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])
candidate = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=250, max_depth=4, learning_rate=.05, subsample=.85, colsample_bytree=.85,
        objective='reg:squarederror', eval_metric='mae', random_state=RANDOM_STATE, n_jobs=-1
    )),
])
candidate.fit(X_train, y_train)
importance = permutation_importance(
    candidate, X_validation, y_validation, n_repeats=5, random_state=RANDOM_STATE,
    scoring='neg_mean_absolute_error', n_jobs=-1
)
importance_df = pd.DataFrame({
    'feature': FEATURES, 'mae_increase': importance.importances_mean,
    'std': importance.importances_std,
}).sort_values('mae_increase', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance_df, y='feature', x='mae_increase', color='#457b9d', ax=ax)
ax.errorbar(importance_df['mae_increase'], np.arange(len(importance_df)),
            xerr=importance_df['std'], fmt='none', ecolor='black', capsize=3)
ax.set(xlabel='Increase in validation MAE after permutation (RWF/kg)', ylabel='Original input feature')
add_title(fig, 'Figure 6. Validation permutation importance for XGBoost candidate')
save_figure(fig, '06_feature_importance.png')
display(importance_df)

## 9. Reproducibility summary

Record the following values in the Phase 2 Decision Log: data date range, split cutoffs, quarantined conflict count, eligible-series count, PCA cutoffs, and any decision to retain or transform skewed variables. The held-out test set remains untouched until model selection is frozen.

In [ ]:
summary = {
    'raw_rows': len(raw),
    'clean_rows': len(clean),
    'conflicting_rows_quarantined': len(conflicts),
    'date_start': clean['observed_date'].min().date(),
    'date_end': clean['observed_date'].max().date(),
    'train_end': train_end.date(),
    'validation_end': validation_end.date(),
    'eligible_crop_market_horizon_series': len(eligible_index),
    'train_examples_after_purge': len(train),
    'validation_examples_after_purge': len(validation),
    'test_examples_after_purge': len(test),
}
pd.Series(summary, name='value').to_frame()